# Card Configuration Guide

This notebook shows how to configure the pinned card for a real molecule map.

We use `cluster_65053.csv`, molecular properties, SMILES, and Murcko scaffolds.
That is much closer to how most chemistry users will build cards in practice.


## 1. Load Molecules And Build Metadata


In [10]:
from urllib.parse import quote

import pandas as pd

from tmap import TMAP
from tmap.utils import fingerprints_from_smiles, molecular_properties, murcko_scaffolds

df = pd.read_csv("../examples/cluster_65053.csv", nrows=1500)
smiles = df["smiles"].tolist()
cluster_ids = df["cluster_id2"].astype(str).tolist()
molecule_ids = [f"Mol-{i:05d}" for i in range(len(smiles))]

fps = fingerprints_from_smiles(smiles, fp_type="morgan", radius=2, n_bits=2048)
props = molecular_properties(smiles, properties=["mw", "logp", "n_rings", "qed"])
scaffolds = murcko_scaffolds(smiles).tolist()
pubchem_urls = [f"https://pubchem.ncbi.nlm.nih.gov/#query={quote(s)}" for s in smiles]

model = TMAP(metric="jaccard", n_neighbors=20, kc=50, seed=42).fit(fps)
print(f"Fitted: {model.embedding_.shape[0]} molecules")


  [Props] 1,500 done, 0 invalid
  [Scaffolds] 1,500/1,500 valid
Fitted: 1500 molecules


In [11]:
def make_viz(include_urls: bool = False):
    viz = model.to_tmapviz()
    viz.title = "Card Configuration Demo"
    viz.add_smiles(smiles)
    viz.add_color_layout("Cluster", cluster_ids, categorical=True, color="Set2")
    viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
    viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
    viz.add_color_layout("Ring Count", props["n_rings"].tolist(), categorical=True, color="tab10")
    viz.add_color_layout("QED", props["qed"].tolist(), color="magma")
    viz.add_label("Molecule ID", molecule_ids)
    viz.add_label("Murcko Scaffold", scaffolds)
    if include_urls:
        viz.add_label("PubChem URL", pubchem_urls)
    viz.configure_column("smiles", display_name="SMILES", copyable=True)
    viz.configure_column("Murcko Scaffold", copyable=True)
    return viz


## 2. Default Card

Without `configure_card()`, the card shows the label columns that were added to the visualization.


In [12]:
viz = make_viz()
viz.write_html("card_1_default.html")


PosixPath('card_1_default.html')

## 3. Title Only

Use a short molecule identifier as the card title.


In [13]:
viz = make_viz()
viz.configure_card(title_column="Molecule ID")
viz.write_html("card_2_title.html")


PosixPath('card_2_title.html')

## 4. Title And Subtitle

A chemistry-friendly choice is an ID for the title and the scaffold for the subtitle.


In [14]:
viz = make_viz()
viz.configure_card(
    title_column="Molecule ID",
    subtitle_column="Murcko Scaffold",
)
viz.write_html("card_3_title_subtitle.html")


PosixPath('card_3_title_subtitle.html')

## 5. Restrict The Fields

Cards are often clearer when you only show the properties that matter.


In [15]:
viz = make_viz()
viz.configure_card(
    title_column="Molecule ID",
    subtitle_column="Murcko Scaffold",
    fields=["Cluster", "MW", "LogP", "Ring Count", "QED"],
)
viz.write_html("card_4_fields.html")


PosixPath('card_4_fields.html')

## 6. Add Links

Here we add a per-molecule PubChem search button.


In [16]:
viz = make_viz(include_urls=True)
viz.configure_card(
    title_column="Molecule ID",
    subtitle_column="Murcko Scaffold",
    fields=["Cluster", "MW", "LogP", "QED"],
    links=[
        {"label": "PubChem Search", "url": "{PubChem URL}"},
    ],
)
viz.write_html("card_5_links.html")


PosixPath('card_5_links.html')

## 7. Full Chemistry Card

This is a good default for a real molecule map.


In [17]:
viz = make_viz(include_urls=True)
viz.configure_card(
    title_column="Molecule ID",
    subtitle_column="Murcko Scaffold",
    fields=["Cluster", "MW", "LogP", "Ring Count", "QED", "smiles"],
    links=[
        {"label": "PubChem Search", "url": "{PubChem URL}"},
    ],
)
viz.write_html("card_6_full.html")


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_81838/959893289.py:2: UserWarning: configure_card references columns not yet added: ['smiles']. Add them before calling to_html() or the card will be incomplete.
  viz.configure_card(


PosixPath('card_6_full.html')

## Practical Advice

For molecule maps, a good card usually has:

- a short title such as an internal ID
- a scaffold or series label as subtitle
- a small set of important properties
- one or two useful external links
- SMILES shown as text only when you really need it, because the structure tooltip already renders the molecule
